# 01 — Formatação dos datasets e seleção do treino por Cochran

**Extensão do paper "Leveraging LLM Reflection to Improve Small Language Model Agents' Capabilities"**

Este notebook faz duas coisas, nesta ordem:

1. **Formatação.** Lê os cinco datasets puros de `data/raw/` e os reescreve num
   schema MCQ único em `data/processed/`. Depois deste passo, nenhum código do
   pipeline precisa saber de onde a questão veio.
2. **Seleção.** Aplica a fórmula de Cochran (95% de confiança, margem de 5
   pontos) ao **split de treino** de cada dataset e grava a amostra em
   `data/splits/`. Validação e teste passam inteiros, sem seleção — são as
   partições oficiais contra as quais o paper reporta.

Pré-requisito: `python scripts/setup_datasets.py`.

In [1]:
import json
import re
import sys
from collections import Counter
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))

from config import (
    CHOICE_LABELS,
    COCHRAN_CONFIDENCE,
    COCHRAN_FINITE_CORRECTION,
    COCHRAN_MARGIN,
    COCHRAN_PROPORTION,
    COCHRAN_STRATIFY_BY,
    DATASETS,
    PROCESSED_DIR,
    RAW_DIR,
    SEED,
    SPLITS_DIR,
    ensure_dirs,
)
from common import (
    build_answer_prompt,
    cochran_sample_size,
    label_distribution,
    label_invariant_report,
    read_jsonl,
    stratified_sample,
    validate_mcq_item,
    write_jsonl,
)

ensure_dirs()
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)      # senão as tabelas de rótulos truncam
pd.set_option("display.max_columns", 30)

print(f"raiz do projeto : {ROOT}")
print(f"datasets puros  : {RAW_DIR}")
print(f"saída formatada : {PROCESSED_DIR}")
print(f"saída do treino : {SPLITS_DIR}")

raiz do projeto : /home/rodrigo.flexa/Reflection-MCQ
datasets puros  : /home/rodrigo.flexa/Reflection-MCQ/data/raw
saída formatada : /home/rodrigo.flexa/Reflection-MCQ/data/processed
saída do treino : /home/rodrigo.flexa/Reflection-MCQ/data/splits


## Schema unificado

Cada item, em qualquer dataset, vira um dicionário com estes campos:

| campo | tipo | descrição |
|---|---|---|
| `uid` | str | `<dataset>-<split>-<índice>`, estável entre execuções |
| `dataset` | str | chave em `config.DATASETS` |
| `split` | str | `train`, `validation` ou `test` |
| `problem_type` | str | `process` (raciocínio) ou `knowledge` (fato) |
| `context` | str \| None | premissa do LogiQA2, fato de apoio do OpenBookQA, senão `None` |
| `question` | str | enunciado |
| `choices` | list | `[{"label": "A", "text": "..."}, ...]`, rótulos sempre A, B, C, ... |
| `answerKey` | str | letra correta |
| `num_choices` | int | 4 em quase tudo, 5 no AQuA |
| `rationale` | str \| None | solução de referência quando o dataset fornece |
| `source_id` | str | identificador original, para rastrear de volta |

Duas decisões que valem registrar:

- **Rótulos são renumerados para A, B, C, ...** Parte do ARC usa `1..4`. Manter
  os dois formatos obrigaria o extrator a aceitar dígitos, e aí qualquer número
  solto no raciocínio se tornaria um candidato a resposta.
- **`context` fica separado de `question`.** A etapa de recuperação por
  similaridade compara enunciados; misturar uma premissa de 80 palavras do
  LogiQA2 dentro do enunciado distorceria o embedding. A junção para o prompt
  acontece em `common.format_question`, num só lugar.

In [2]:
def make_uid(dataset: str, split: str, idx: int) -> str:
    return f"{dataset}-{split}-{idx:06d}"


def build_choices(texts, labels=None):
    """Monta a lista de alternativas com rótulos canônicos A, B, C, ...

    Se `labels` vier do dataset original, ele só é usado para localizar o
    gabarito por posição; o rótulo gravado é sempre o canônico.
    """
    texts = [str(t).strip() for t in texts]
    if len(texts) > len(CHOICE_LABELS):
        raise ValueError(f"{len(texts)} alternativas excede o alfabeto suportado")
    return [{"label": CHOICE_LABELS[i], "text": t} for i, t in enumerate(texts)]


def canonical_answer(original_labels, answer_key) -> str:
    """Traduz o gabarito original para a letra canônica, pela posição."""
    labels = [str(l).strip().upper() for l in original_labels]
    key = str(answer_key).strip().upper()
    if key not in labels:
        raise ValueError(f"gabarito {key!r} não está entre os rótulos {labels}")
    return CHOICE_LABELS[labels.index(key)]


def read_raw(dataset: str, split: str) -> pd.DataFrame:
    path = RAW_DIR / dataset / f"{split}.parquet"
    if not path.exists():
        raise FileNotFoundError(
            f"{path} não existe. Rode: python scripts/setup_datasets.py --datasets {dataset}"
        )
    return pd.read_parquet(path)


# Registro dos normalizadores; preenchido nas células seguintes.
NORMALIZERS = {}

## ARC-Challenge e OpenBookQA

Os dois já são MCQ nativos e vêm com `choices` como struct de `text` e `label`.
Diferenças a tratar:

- **ARC:** uma fração dos itens usa rótulos numéricos (`"1".."4"`) e um punhado
  tem 3 ou 5 alternativas em vez de 4. Não descartamos os que fogem de 4 —
  `num_choices` fica registrado e a acurácia do chute é calculada por item.
- **OpenBookQA:** o enunciado está em `question_stem`. A configuração `main` não
  traz o fato de apoio (isso é a `additional`), então `context` é `None` — é de
  propósito: queremos medir conhecimento do modelo, não leitura de um fato dado.

In [3]:
def normalize_arc(df: pd.DataFrame, split: str) -> tuple[list[dict], list[str]]:
    items, skipped = [], []
    for idx, row in enumerate(df.itertuples(index=False)):
        ch = row.choices
        try:
            choices = build_choices(ch["text"])
            answer = canonical_answer(ch["label"], row.answerKey)
        except Exception as exc:
            skipped.append(f"{row.id}: {exc}")
            continue
        items.append({
            "uid": make_uid("arc", split, idx),
            "dataset": "arc",
            "split": split,
            "problem_type": DATASETS["arc"].problem_type,
            "context": None,
            "question": str(row.question).strip(),
            "choices": choices,
            "answerKey": answer,
            "num_choices": len(choices),
            "rationale": None,
            "source_id": str(row.id),
        })
    return items, skipped


def normalize_openbookqa(df: pd.DataFrame, split: str) -> tuple[list[dict], list[str]]:
    items, skipped = [], []
    for idx, row in enumerate(df.itertuples(index=False)):
        ch = row.choices
        try:
            choices = build_choices(ch["text"])
            answer = canonical_answer(ch["label"], row.answerKey)
        except Exception as exc:
            skipped.append(f"{row.id}: {exc}")
            continue
        items.append({
            "uid": make_uid("openbookqa", split, idx),
            "dataset": "openbookqa",
            "split": split,
            "problem_type": DATASETS["openbookqa"].problem_type,
            "context": None,
            "question": str(row.question_stem).strip(),
            "choices": choices,
            "answerKey": answer,
            "num_choices": len(choices),
            "rationale": None,
            "source_id": str(row.id),
        })
    return items, skipped


NORMALIZERS["arc"] = normalize_arc
NORMALIZERS["openbookqa"] = normalize_openbookqa

## AQuA-RAT e LogiQA 2.0

- **AQuA:** cinco alternativas, no formato `["A)1/2", "B)3", ...]`. O `rationale`
  do dataset é guardado, mas nunca entra no prompt do aluno — ele existe para a
  análise de erro depois.
- **LogiQA2:** `answer` é índice inteiro, `text` é a premissa (vai para
  `context`) e `question` é a pergunta sobre ela.

In [4]:
AQUA_OPTION = re.compile(r"^\s*\(?([A-Ea-e])\)?\s*[\)\.\:\-]?\s*(.+)$", re.DOTALL)


def parse_aqua_options(options) -> list[str]:
    """['A)1/2', 'B)3', ...] -> ['1/2', '3', ...], validando a ordem A..E."""
    texts = []
    for pos, raw in enumerate(options):
        m = AQUA_OPTION.match(str(raw))
        if not m:
            raise ValueError(f"opção sem rótulo reconhecível: {raw!r}")
        letter, text = m.group(1).upper(), m.group(2).strip()
        if letter != CHOICE_LABELS[pos]:
            raise ValueError(f"opções fora de ordem: esperava {CHOICE_LABELS[pos]}, veio {letter}")
        if not text:
            raise ValueError(f"opção vazia: {raw!r}")
        texts.append(text)
    return texts


def normalize_aqua(df: pd.DataFrame, split: str) -> tuple[list[dict], list[str]]:
    items, skipped = [], []
    for idx, row in enumerate(df.itertuples(index=False)):
        try:
            texts = parse_aqua_options(row.options)
            choices = build_choices(texts)
            answer = canonical_answer(CHOICE_LABELS[: len(texts)], row.correct)
        except Exception as exc:
            skipped.append(f"aqua[{idx}]: {exc}")
            continue
        items.append({
            "uid": make_uid("aqua", split, idx),
            "dataset": "aqua",
            "split": split,
            "problem_type": DATASETS["aqua"].problem_type,
            "context": None,
            "question": str(row.question).strip(),
            "choices": choices,
            "answerKey": answer,
            "num_choices": len(choices),
            "rationale": str(row.rationale).strip() or None,
            "source_id": f"aqua-{split}-{idx}",
        })
    return items, skipped


def normalize_logiqa2(df: pd.DataFrame, split: str) -> tuple[list[dict], list[str]]:
    items, skipped = [], []
    for idx, row in enumerate(df.itertuples(index=False)):
        try:
            choices = build_choices(list(row.options))
            answer_idx = int(row.answer)
            if not 0 <= answer_idx < len(choices):
                raise ValueError(f"índice de resposta {answer_idx} fora de {len(choices)} opções")
            answer = CHOICE_LABELS[answer_idx]
        except Exception as exc:
            skipped.append(f"logiqa2[{idx}]: {exc}")
            continue
        items.append({
            "uid": make_uid("logiqa2", split, idx),
            "dataset": "logiqa2",
            "split": split,
            "problem_type": DATASETS["logiqa2"].problem_type,
            "context": str(row.text).strip() or None,
            "question": str(row.question).strip(),
            "choices": choices,
            "answerKey": answer,
            "num_choices": len(choices),
            "rationale": None,
            "source_id": f"logiqa2-{row.id}",
        })
    return items, skipped


NORMALIZERS["aqua"] = normalize_aqua
NORMALIZERS["logiqa2"] = normalize_logiqa2

## GSM8K: conversão para múltipla escolha

**Este é o único dataset que não é MCQ na origem** — a resposta é um número
aberto. Como todo o desenho experimental (extrator, métrica de virada, taxa de
chute) pressupõe alternativas, o GSM8K precisa ser convertido, e a escolha dos
distratores muda o quão difícil o item fica.

Distratores aleatórios seriam fáceis demais: qualquer estimativa de ordem de
grandeza eliminaria três das quatro opções, e a acurácia mediria aritmética
grosseira, não raciocínio multi-passo.

A solução usa uma propriedade do próprio dataset: as soluções do GSM8K vêm com
os passos aritméticos anotados como `<<48/2=24>>`. Esses valores intermediários
são exatamente onde um modelo que erra o número de passos costuma parar. Então:

1. o gabarito é o número depois de `####`;
2. os candidatos a distrator são os **resultados intermediários**, na ordem
   inversa — o penúltimo passo é o erro mais plausível de todos;
3. se sobrarem menos de três, completamos com perturbações típicas (dobro,
   metade, off-by-one, erro de escala por 10);
4. as quatro opções são embaralhadas com semente derivada do enunciado, para que
   a posição da resposta correta seja estável entre execuções e não se
   concentre numa letra.

A consequência a registrar no paper: **o GSM8K aqui não é comparável a números
de GSM8K aberto da literatura.** É um GSM8K-MCQ construído por este
procedimento. `rationale` guarda a solução original para auditoria.

In [5]:
CALC_STEP = re.compile(r"<<[^=<>]*=([^<>]+)>>")
FINAL_MARK = re.compile(r"####\s*(.+?)\s*$", re.MULTILINE)


def _to_number(text: str):
    """'1,200' -> 1200.0. Devolve None se não for número limpo."""
    cleaned = str(text).replace(",", "").replace("$", "").replace("%", "").strip()
    try:
        return float(cleaned)
    except ValueError:
        return None


def _fmt(value: float) -> str:
    """Inteiro sem '.0'; caso contrário duas casas, sem zeros à direita."""
    if abs(value - round(value)) < 1e-9:
        return str(int(round(value)))
    return f"{value:.2f}".rstrip("0").rstrip(".")


def gsm8k_distractors(solution: str, gold: float, n: int = 3) -> tuple[list[str], str]:
    """Distratores para um item do GSM8K, e a fonte usada ('steps', 'mixed', 'synthetic')."""
    seen = {_fmt(gold)}
    out = []

    # 1. resultados intermediários anotados, do passo mais tardio para o mais cedo
    steps = [_to_number(v) for v in CALC_STEP.findall(solution)]
    for value in reversed([s for s in steps if s is not None]):
        key = _fmt(value)
        if key not in seen and value >= 0:
            seen.add(key)
            out.append(key)
        if len(out) == n:
            return out, "steps"

    n_from_steps = len(out)

    # 2. perturbações aritméticas típicas, em ordem de plausibilidade
    for value in (gold * 2, gold / 2, gold + 1, gold - 1, gold * 10,
                  gold / 10, gold + 10, gold - 10, gold * 3, gold / 3):
        if value is None or value < 0:
            continue
        key = _fmt(value)
        if key not in seen:
            seen.add(key)
            out.append(key)
        if len(out) == n:
            break

    # 3. último recurso: deslocamentos crescentes
    shift = 2
    while len(out) < n:
        key = _fmt(gold + shift)
        if key not in seen:
            seen.add(key)
            out.append(key)
        shift += 1

    return out, ("mixed" if n_from_steps else "synthetic")


def normalize_gsm8k(df: pd.DataFrame, split: str) -> tuple[list[dict], list[str]]:
    import random

    items, skipped = [], []
    for idx, row in enumerate(df.itertuples(index=False)):
        solution = str(row.answer)
        mark = FINAL_MARK.search(solution)
        if not mark:
            skipped.append(f"gsm8k[{idx}]: sem marcador ####")
            continue
        gold = _to_number(mark.group(1))
        if gold is None:
            skipped.append(f"gsm8k[{idx}]: resposta não numérica {mark.group(1)!r}")
            continue

        distractors, source = gsm8k_distractors(solution, gold)
        pool = [_fmt(gold)] + distractors

        # Semente por item: embaralhamento estável e reproduzível.
        rng = random.Random(f"{SEED}-gsm8k-{split}-{idx}")
        rng.shuffle(pool)

        choices = build_choices(pool)
        answer = CHOICE_LABELS[pool.index(_fmt(gold))]

        items.append({
            "uid": make_uid("gsm8k", split, idx),
            "dataset": "gsm8k",
            "split": split,
            "problem_type": DATASETS["gsm8k"].problem_type,
            "context": None,
            "question": str(row.question).strip(),
            "choices": choices,
            "answerKey": answer,
            "num_choices": len(choices),
            # Solução original, sem as anotações de calculadora.
            "rationale": CALC_STEP.sub("", solution).strip(),
            "source_id": f"gsm8k-{split}-{idx}",
            "distractor_source": source,
        })
    return items, skipped


NORMALIZERS["gsm8k"] = normalize_gsm8k

### Conferência dos distratores do GSM8K

Antes de rodar tudo, uma olhada em alguns itens convertidos. O que queremos ver:
distratores na mesma ordem de grandeza do gabarito e majoritariamente vindos de
`steps`.

In [6]:
_gsm_preview_df = read_raw("gsm8k", "train").head(300)
_gsm_preview, _gsm_skipped = normalize_gsm8k(_gsm_preview_df, "train")

print("fonte dos distratores nos 300 primeiros itens:")
for source, count in Counter(i["distractor_source"] for i in _gsm_preview).most_common():
    print(f"  {source:<10} {count:>4}  ({count / len(_gsm_preview):.1%})")
print(f"  descartados: {len(_gsm_skipped)}")

for item in _gsm_preview[:3]:
    print("\n" + "-" * 78)
    print(item["question"][:260])
    for c in item["choices"]:
        print(f"   {c['label']}) {c['text']}" + ("   <- gabarito" if c["label"] == item["answerKey"] else ""))

fonte dos distratores nos 300 primeiros itens:
  mixed       167  (55.7%)
  steps       120  (40.0%)
  synthetic    13  (4.3%)
  descartados: 0

------------------------------------------------------------------------------
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
   A) 144
   B) 24
   C) 36
   D) 72   <- gabarito

------------------------------------------------------------------------------
Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?
   A) 0.2
   B) 5
   C) 10   <- gabarito
   D) 20

------------------------------------------------------------------------------
Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to 

## Formatação de todos os datasets

Cada split passa pelo normalizador, é validado item a item por
`common.validate_mcq_item` e gravado em `data/processed/<dataset>/<split>.jsonl`.
Itens inválidos são descartados e contados. Espere descartes reais: o AQuA tem
cerca de 2.500 itens de treino com duas alternativas de texto idêntico, o que
torna o gabarito ambíguo, e o ARC e o OpenBookQA têm um punhado de itens com
alternativa vazia.

In [7]:
processing_report = []
processed_counts = {}

for key, spec in DATASETS.items():
    normalize = NORMALIZERS[key]
    for split in spec.splits:
        df = read_raw(key, split)
        items, skipped = normalize(df, split)

        invalid = []
        valid = []
        for item in items:
            problems = validate_mcq_item(item)
            if problems:
                invalid.append((item["uid"], problems))
            else:
                valid.append(item)

        out_path = PROCESSED_DIR / key / f"{split}.jsonl"
        write_jsonl(out_path, valid)
        processed_counts[(key, split)] = len(valid)

        processing_report.append({
            "dataset": key,
            "split": split,
            "brutos": len(df),
            "válidos": len(valid),
            "descartados": len(skipped),
            "inválidos": len(invalid),
            "n_choices": sorted({i["num_choices"] for i in valid}),
        })

        if skipped[:2] or invalid[:2]:
            print(f"[{key}/{split}] exemplos de problema:")
            for s in skipped[:2]:
                print(f"    descartado: {s}")
            for uid, problems in invalid[:2]:
                print(f"    inválido {uid}: {problems}")

report_df = pd.DataFrame(processing_report)
report_df

[aqua/train] exemplos de problema:
    inválido aqua-train-000024: ['alternativas duplicadas']
    inválido aqua-train-000053: ['alternativas duplicadas']
[aqua/validation] exemplos de problema:
    inválido aqua-validation-000092: ['alternativas duplicadas']
    inválido aqua-validation-000190: ['alternativas duplicadas']
[aqua/test] exemplos de problema:
    inválido aqua-test-000013: ['alternativas duplicadas']
    inválido aqua-test-000117: ['alternativas duplicadas']
[logiqa2/train] exemplos de problema:
    inválido logiqa2-train-000043: ['alternativas duplicadas']
    inválido logiqa2-train-000235: ['alternativas duplicadas']
[logiqa2/validation] exemplos de problema:
    inválido logiqa2-validation-000020: ['alternativas duplicadas']
    inválido logiqa2-validation-000060: ['alternativas duplicadas']
[logiqa2/test] exemplos de problema:
    inválido logiqa2-test-000400: ['alternativas duplicadas']
    inválido logiqa2-test-000797: ['alternativas duplicadas']
[arc/train] exemplo

,dataset,split,brutos,válidos,descartados,inválidos,n_choices
0,gsm8k,train,7473,7473,0,0,[4]
1,gsm8k,test,1319,1319,0,0,[4]
2,aqua,train,97467,94939,0,2528,[5]
3,aqua,validation,254,252,0,2,[5]
4,aqua,test,254,246,0,8,[5]
5,logiqa2,train,12567,12537,0,30,[4]
6,logiqa2,validation,1569,1565,0,4,[4]
7,logiqa2,test,1572,1565,0,7,[4]
8,arc,train,1119,1117,0,2,"[3, 4, 5]"
9,arc,validation,299,298,0,1,"[3, 4, 5]"


## Vazamento entre treino e teste

Este passo não estava previsto no Caderno e é o mais importante deste notebook.

A premissa central da extensão é que separar treino de teste transforma
memorização em generalização: a reflexão gerada sobre uma questão de treino é
recuperada para uma questão *diferente*, mas semanticamente próxima, no teste.
A limitação do paper anterior era exatamente reaplicar a reflexão na mesma
questão que a gerou.

Só que os splits oficiais não garantem que as questões sejam distintas. Se um
enunciado aparecer nos dois lados, a recuperação por similaridade vai encontrá-lo
com cosseno igual a 1 e devolver a reflexão da questão idêntica — reproduzindo o
confundidor que a extensão foi desenhada para eliminar, e sem deixar rastro nos
resultados.

A célula abaixo mede isso. Vale ler os números antes de decidir qualquer coisa.

In [8]:
def dedup_key(item: dict) -> str:
    """Chave de identidade de uma questão: enunciado + contexto, normalizados."""
    text = f"{item.get('context') or ''} || {item['question']}"
    return re.sub(r"\W+", " ", text.lower()).strip()


leak_report = []
for key, spec in DATASETS.items():
    train_items = read_jsonl(PROCESSED_DIR / key / "train.jsonl")

    held_out = set()
    for split in spec.splits:
        if split == "train":
            continue
        held_out |= {dedup_key(i) for i in read_jsonl(PROCESSED_DIR / key / f"{split}.jsonl")}

    n_leaked = sum(1 for i in train_items if dedup_key(i) in held_out)
    internal = len(train_items) - len({dedup_key(i) for i in train_items})

    leak_report.append({
        "dataset": key,
        "treino": len(train_items),
        "também em val/teste": n_leaked,
        "% vazado": f"{n_leaked / len(train_items):.1%}",
        "duplicatas internas": internal,
    })

leak_df = pd.DataFrame(leak_report)
leak_df

,dataset,treino,também em val/teste,% vazado,duplicatas internas
0,gsm8k,7473,0,0.0%,0
1,aqua,94939,44,0.0%,17920
2,logiqa2,12537,449,3.6%,934
3,arc,1117,7,0.6%,1
4,openbookqa,4954,45,0.9%,129


O LogiQA2 é o caso grave: uma fatia grande do treino reaparece em validação ou
teste, porque o dataset reutiliza a mesma premissa em vários itens e às vezes
repete o par premissa mais pergunta entre splits. O AQuA tem muita duplicata
interna, efeito de problemas gerados por template com números trocados.

A política adotada, e a razão de cada parte:

- **Validação e teste passam intactos.** São as partições oficiais contra as
  quais o paper reporta; mexer nelas quebra a comparabilidade com a literatura.
- **O treino é deduplicado antes da amostragem de Cochran**, removendo tanto o
  que colide com validação e teste quanto as repetições internas. O treino é
  nosso para escolher, então o custo cai todo aqui.
- **A deduplicação vem antes de Cochran, não depois**, para que o $N$ da fórmula
  seja a população limpa de onde a amostra realmente sai.

O custo é quase nulo: o total de treino cai de 1.766 para 1.763 questões, porque
com $p = 0.5$ o $n$ de Cochran é praticamente insensível a $N$ nessa faixa. Ou
seja, a correção do confundidor sai de graça.

In [9]:
train_pool = {}
dedup_report = []

for key, spec in DATASETS.items():
    train_items = read_jsonl(PROCESSED_DIR / key / "train.jsonl")

    held_out = set()
    for split in spec.splits:
        if split == "train":
            continue
        held_out |= {dedup_key(i) for i in read_jsonl(PROCESSED_DIR / key / f"{split}.jsonl")}

    seen = set()
    pool = []
    for item in train_items:
        k = dedup_key(item)
        if k in held_out or k in seen:
            continue
        seen.add(k)
        pool.append(item)

    train_pool[key] = pool
    dedup_report.append({
        "dataset": key,
        "N bruto": len(train_items),
        "removido": len(train_items) - len(pool),
        "N limpo": len(pool),
    })

dedup_df = pd.DataFrame(dedup_report)

# Nenhuma questão do pool de treino pode aparecer em validação ou teste.
for key, spec in DATASETS.items():
    held_out = set()
    for split in spec.splits:
        if split == "train":
            continue
        held_out |= {dedup_key(i) for i in read_jsonl(PROCESSED_DIR / key / f"{split}.jsonl")}
    overlap = [i["uid"] for i in train_pool[key] if dedup_key(i) in held_out]
    assert not overlap, f"{key}: {len(overlap)} itens ainda vazam"

print("verificado: nenhuma questão do pool de treino aparece em validação ou teste")
dedup_df

verificado: nenhuma questão do pool de treino aparece em validação ou teste


,dataset,N bruto,removido,N limpo
0,gsm8k,7473,0,7473
1,aqua,94939,17947,76992
2,logiqa2,12537,1369,11168
3,arc,1117,8,1109
4,openbookqa,4954,164,4790


## Seleção do treino pela fórmula de Cochran

$$n_0 = \frac{z^2\,p\,(1-p)}{e^2}
\qquad\qquad
n = \frac{n_0}{1 + \dfrac{n_0 - 1}{N}}$$

Com $z = 1.96$ (95%), $e = 0.05$ e $p = 0.5$, temos $n_0 \approx 385$. O valor
$p = 0.5$ é o que maximiza a variância, então $n_0$ é o caso conservador: não
precisamos saber a acurácia esperada de antemão.

A segunda fórmula é a correção para população finita. Ela importa aqui porque os
treinos têm tamanhos muito diferentes — de cerca de 1.100 no ARC a 77.000 no
AQuA depois da deduplicação — e o esforço de amostragem só precisa crescer com a
população até certo ponto. Para o ARC a correção reduz de 385 para 286; para o
AQuA quase não muda nada.

O $N$ usado é a **população deduplicada** da célula anterior, não o treino bruto.

**Isto substitui os tamanhos registrados antes no Caderno** (708 no ARC, 3.806 no
OpenBookQA, 5.611 no GSM8K, 9.360 no LogiQA2, 5.244 no AQuA). Aqueles números não
vinham de Cochran e implicariam cerca de 31 mil questões de treino por aluno,
multiplicadas por professor, profundidade de reflexão e condição. Com Cochran o
treino cai para menos de 1.800 questões no total.

A amostragem é **estratificada pelo gabarito**: se a proporção de respostas "A"
mudar entre o dataset completo e a amostra, a acurácia esperada de um chute muda
com ela, e a comparação com o baseline fica contaminada.

In [10]:
cochran_rows = []
for key in DATASETS:
    n_train = len(train_pool[key])  # população limpa, pós-deduplicação
    stats = cochran_sample_size(
        population=n_train,
        confidence=COCHRAN_CONFIDENCE,
        margin=COCHRAN_MARGIN,
        proportion=COCHRAN_PROPORTION,
        finite_correction=COCHRAN_FINITE_CORRECTION,
    )
    sem_correcao = cochran_sample_size(
        population=n_train, confidence=COCHRAN_CONFIDENCE,
        margin=COCHRAN_MARGIN, proportion=COCHRAN_PROPORTION, finite_correction=False,
    )["n"]
    cochran_rows.append({
        "dataset": key,
        "N (treino)": n_train,
        "n0": stats["n0"],
        "n sem correção": sem_correcao,
        "n final": stats["n"],
        "% de N": f"{stats['fraction_of_population']:.1%}",
    })

cochran_df = pd.DataFrame(cochran_rows)
print(f"z = {cochran_sample_size(1000)['z']}   (confiança {COCHRAN_CONFIDENCE:.0%}, margem {COCHRAN_MARGIN:.0%}, p = {COCHRAN_PROPORTION})")
print(f"total de treino após seleção: {cochran_df['n final'].sum():,} questões")
cochran_df

z = 1.959964   (confiança 95%, margem 5%, p = 0.5)
total de treino após seleção: 1,763 questões


,dataset,N (treino),n0,n sem correção,n final,% de N
0,gsm8k,7473,385,385,366,4.9%
1,aqua,76992,385,385,383,0.5%
2,logiqa2,11168,385,385,372,3.3%
3,arc,1109,385,385,286,25.8%
4,openbookqa,4790,385,385,356,7.4%


In [11]:
selection_report = []
selected_by_dataset = {}

for key in DATASETS:
    train_items = train_pool[key]
    n = cochran_sample_size(
        population=len(train_items),
        confidence=COCHRAN_CONFIDENCE,
        margin=COCHRAN_MARGIN,
        proportion=COCHRAN_PROPORTION,
        finite_correction=COCHRAN_FINITE_CORRECTION,
    )["n"]

    sample = stratified_sample(train_items, n=n, stratify_by=COCHRAN_STRATIFY_BY, seed=SEED)
    selected_by_dataset[key] = sample

    write_jsonl(SPLITS_DIR / key / "train.jsonl", sample)

    # Validação e teste seguem inteiros; copiados para splits/ para que o
    # restante do pipeline leia sempre do mesmo lugar.
    for split in DATASETS[key].splits:
        if split == "train":
            continue
        write_jsonl(SPLITS_DIR / key / f"{split}.jsonl",
                    read_jsonl(PROCESSED_DIR / key / f"{split}.jsonl"))

    pop_dist = label_distribution(train_items)
    smp_dist = label_distribution(sample)
    max_drift = max(abs(smp_dist.get(k, 0) - v) for k, v in pop_dist.items())

    selection_report.append({
        "dataset": key,
        "N": len(train_items),
        "n": len(sample),
        "dist. população": pop_dist,
        "dist. amostra": smp_dist,
        "desvio máx.": f"{max_drift:.3f}",
    })

selection_df = pd.DataFrame(selection_report)
selection_df

,dataset,N,n,dist. população,dist. amostra,desvio máx.
0,gsm8k,7473,366,"{'A': 0.2529, 'B': 0.2542, 'C': 0.2426, 'D': 0.2502}","{'A': 0.2514, 'B': 0.2541, 'C': 0.2432, 'D': 0.2514}",0.002
1,aqua,76992,383,"{'A': 0.2049, 'B': 0.2128, 'C': 0.2231, 'D': 0.2032, 'E': 0.156}","{'A': 0.2037, 'B': 0.2141, 'C': 0.2219, 'D': 0.2037, 'E': 0.1567}",0.001
2,logiqa2,11168,372,"{'A': 0.2261, 'B': 0.2454, 'C': 0.263, 'D': 0.2655}","{'A': 0.2258, 'B': 0.2446, 'C': 0.2634, 'D': 0.2661}",0.001
3,arc,1109,286,"{'A': 0.2137, 'B': 0.2642, 'C': 0.2597, 'D': 0.2624}","{'A': 0.2133, 'B': 0.2657, 'C': 0.2587, 'D': 0.2622}",0.002
4,openbookqa,4790,356,"{'A': 0.2772, 'B': 0.2453, 'C': 0.2303, 'D': 0.2472}","{'A': 0.2781, 'B': 0.2444, 'C': 0.2303, 'D': 0.2472}",0.001


## Padronização das alternativas

Este é o ponto em que os arquivos de `data/splits/` acabaram de ser escritos, e
é onde o invariante central do formato precisa ser verificado, não assumido.

**O invariante: todo item, em todo dataset, usa rótulos de letra contíguos
começando em A.** `A, B, C, ...`, sempre, independentemente do que o dataset
original usava. Três dos cinco chegaram sem isso:

| Dataset | Como vinha | Como fica |
|---|---|---|
| ARC | parte dos itens com rótulos `"1".."4"` | `A, B, C, D` |
| AQuA | opções como `["A)5", "B)10", ...]` | rótulo separado do texto: `A` + `5` |
| LogiQA2 | `answer` como índice inteiro `0..3` | `A, B, C, D` |
| GSM8K | não era múltipla escolha | `A, B, C, D` após gerar distratores |
| OpenBookQA | já em letras | inalterado |

A tradução é sempre **por posição**, nunca pelo rótulo textual do original. Isso
importa no ARC: um item que vem com `["1","2","3","4"]` e gabarito `"3"` vira
gabarito `C` porque `"3"` é o terceiro rótulo, não porque `3` pareça com `C`.

**O número de alternativas pode variar, e isso não é problema.** O AQuA tem 5 em
todos os itens; uma fração pequena do ARC tem 3 ou 5. O invariante é sobre a
FORMA do rótulo, não sobre a quantidade, e as duas partes do pipeline que
dependem disso já tratam a variação item a item:

- **o extrator** monta o conjunto de letras aceitas a partir das alternativas de
  cada item, então um item de 3 alternativas nunca aceita `D`;
- **a análise** pondera a acurácia do chute por `num_choices`, item a item, em
  vez de assumir 25% para todo mundo.

O que quebraria as duas coisas é um rótulo numérico ou fora de ordem: aí qualquer
número solto no raciocínio se tornaria candidato a resposta. Por isso o invariante
é verificado com `assert` abaixo, e não apenas relatado.

In [12]:
standard_rows = []
failures = []

for key, spec in DATASETS.items():
    for split in spec.splits:
        path = SPLITS_DIR / key / f"{split}.jsonl"
        if not path.exists():
            continue
        rep = label_invariant_report(read_jsonl(path))
        if not rep["ok"]:
            failures.append((key, split, rep["violations"][:3]))
        standard_rows.append({
            "dataset": key,
            "split": split,
            "n": rep["n"],
            "rótulos": ", ".join(sorted(rep["label_patterns"])),
            "n_alternativas": rep["num_choices"],
            "uniforme": rep["uniform"],
            "acurácia do chute": rep["chance_accuracy"],
            "ok": rep["ok"],
        })

for key, split, viols in failures:
    print(f"[{key}/{split}] VIOLAÇÕES: {viols}")

assert not failures, (
    "invariante de rótulos violado. Nenhum arquivo de data/splits/ pode sair "
    "com rótulo numérico ou fora de ordem."
)
print("invariante verificado: todos os itens usam rótulos de letra contíguos a partir de A")

standard_df = pd.DataFrame(standard_rows)
standard_df

invariante verificado: todos os itens usam rótulos de letra contíguos a partir de A


,dataset,split,n,rótulos,n_alternativas,uniforme,acurácia do chute,ok
0,gsm8k,train,366,ABCD,{4: 366},True,0.2500,True
1,gsm8k,test,1319,ABCD,{4: 1319},True,0.2500,True
2,aqua,train,383,ABCDE,{5: 383},True,0.2000,True
3,aqua,validation,252,ABCDE,{5: 252},True,0.2000,True
4,aqua,test,246,ABCDE,{5: 246},True,0.2000,True
5,logiqa2,train,372,ABCD,{4: 372},True,0.2500,True
6,logiqa2,validation,1565,ABCD,{4: 1565},True,0.2500,True
7,logiqa2,test,1565,ABCD,{4: 1565},True,0.2500,True
8,arc,train,286,ABCD,{4: 286},True,0.2500,True
9,arc,validation,298,"ABC, ABCD, ABCDE","{3: 3, 4: 294, 5: 1}",False,0.2507,True


In [13]:
# O extrator aceita só as letras do próprio item. Confirmado nos itens que
# fogem de 4 alternativas, que são justamente onde um extrator descuidado
# inventaria uma resposta fora do intervalo.
from common import extract_final_answer

odd = [
    i for key in DATASETS
    for split in DATASETS[key].splits
    if (SPLITS_DIR / key / f"{split}.jsonl").exists()
    for i in read_jsonl(SPLITS_DIR / key / f"{split}.jsonl")
    if i["num_choices"] != 4
]
print(f"{len(odd)} itens com número de alternativas diferente de 4")
for item in odd[:4]:
    labels = [c["label"] for c in item["choices"]]
    beyond = CHOICE_LABELS[len(labels)]  # primeira letra inválida para este item
    probe = extract_final_answer(f"FINAL ANSWER: {beyond}", item["choices"])
    valid = extract_final_answer(f"FINAL ANSWER: {labels[-1]}", item["choices"])
    print(
        f"  {item['uid']:<26} {len(labels)} alternativas ({''.join(labels)})  "
        f"chute={1 / len(labels):.3f}  "
        f"'{beyond}' -> {probe.letter}  '{labels[-1]}' -> {valid.letter}"
    )
    assert probe.letter is None, f"{item['uid']}: extrator aceitou letra fora do item"
    assert valid.letter == labels[-1]
print("\nconfirmado: letra fora do intervalo do item é rejeitada, não contada como erro")

892 itens com número de alternativas diferente de 4
  aqua-train-000082          5 alternativas (ABCDE)  chute=0.200  'F' -> None  'E' -> E
  aqua-train-000165          5 alternativas (ABCDE)  chute=0.200  'F' -> None  'E' -> E
  aqua-train-000490          5 alternativas (ABCDE)  chute=0.200  'F' -> None  'E' -> E
  aqua-train-000620          5 alternativas (ABCDE)  chute=0.200  'F' -> None  'E' -> E

confirmado: letra fora do intervalo do item é rejeitada, não contada como erro


## Verificações finais

Quatro coisas a confirmar antes de gerar qualquer baseline:

1. os tamanhos gravados batem com os de Cochran;
2. os `uid` são únicos dentro e entre datasets;
3. a distribuição do gabarito na amostra acompanha a da população (desvio
   máximo acima já indica isso);
4. o prompt congelado, renderizado de ponta a ponta, sai legível em todos os
   cinco datasets.

In [14]:
checks = []

all_uids = []
for key in DATASETS:
    for split in DATASETS[key].splits:
        rows = read_jsonl(SPLITS_DIR / key / f"{split}.jsonl")
        all_uids.extend(r["uid"] for r in rows)
        expected = (
            cochran_sample_size(
                population=len(train_pool[key]),
                confidence=COCHRAN_CONFIDENCE, margin=COCHRAN_MARGIN,
                proportion=COCHRAN_PROPORTION, finite_correction=COCHRAN_FINITE_CORRECTION,
            )["n"]
            if split == "train" else processed_counts[(key, split)]
        )
        checks.append({
            "dataset": key, "split": split, "linhas": len(rows),
            "esperado": expected, "ok": len(rows) == expected,
        })

checks_df = pd.DataFrame(checks)
duplicates = [u for u, c in Counter(all_uids).items() if c > 1]

print(f"uids totais: {len(all_uids):,}   duplicados: {len(duplicates)}")
print(f"todos os tamanhos conferem: {bool(checks_df['ok'].all())}")
if duplicates[:5]:
    print(f"exemplos de uid duplicado: {duplicates[:5]}")
checks_df

uids totais: 9,179   duplicados: 0
todos os tamanhos conferem: True


,dataset,split,linhas,esperado,ok
0,gsm8k,train,366,366,True
1,gsm8k,test,1319,1319,True
2,aqua,train,383,383,True
3,aqua,validation,252,252,True
4,aqua,test,246,246,True
5,logiqa2,train,372,372,True
6,logiqa2,validation,1565,1565,True
7,logiqa2,test,1565,1565,True
8,arc,train,286,286,True
9,arc,validation,298,298,True


In [15]:
for key in DATASETS:
    item = read_jsonl(SPLITS_DIR / key / "train.jsonl")[0]
    print("=" * 78)
    print(f"{key}  ({item['problem_type']})  uid={item['uid']}  gabarito={item['answerKey']}")
    print("=" * 78)
    print(build_answer_prompt(item)[:1100])
    print()

gsm8k  (process)  uid=gsm8k-train-000005  gabarito=B
You are answering a multiple-choice question.

Question: Mark has a garden with flowers. He planted plants of three different colors in it. Ten of them are yellow, and there are 80% more of those in purple. There are only 25% as many green flowers as there are yellow and purple flowers. How many flowers does Mark have in his garden?

Options:
A) 7
B) 35
C) 18
D) 28

Instructions:
- Think step by step before answering.
- Choose exactly one option.
- End your response with this exact line, and nothing after it:
FINAL ANSWER: <letter>

aqua  (process)  uid=aqua-train-000082  gabarito=C
You are answering a multiple-choice question.

Question: Two numbers are less than third number by 30% and 37% respectively. How much percent is the second number less than by the first

Options:
A) 8%
B) 9%
C) 10%
D) 11%
E) 12%

Instructions:
- Think step by step before answering.
- Choose exactly one option.
- End your response with this exact line, and

In [16]:
summary = []
for key, spec in DATASETS.items():
    row = {"dataset": key, "tipo": spec.problem_type, "MCQ nativo": spec.native_mcq}
    for split in ("train", "validation", "test"):
        path = SPLITS_DIR / key / f"{split}.jsonl"
        row[split] = len(read_jsonl(path)) if path.exists() else 0
    row["alternativas"] = read_jsonl(SPLITS_DIR / key / "train.jsonl")[0]["num_choices"]
    summary.append(row)

summary_df = pd.DataFrame(summary)[
    ["dataset", "tipo", "MCQ nativo", "alternativas", "train", "validation", "test"]
]

manifest = {
    "cochran": {
        "confidence": COCHRAN_CONFIDENCE,
        "margin": COCHRAN_MARGIN,
        "proportion": COCHRAN_PROPORTION,
        "finite_correction": COCHRAN_FINITE_CORRECTION,
        "stratify_by": COCHRAN_STRATIFY_BY,
        "seed": SEED,
    },
    "deduplication": {
        "policy": (
            "treino deduplicado contra validação e teste e contra si mesmo, "
            "por (contexto + enunciado) normalizado; val e teste intactos"
        ),
        "applied_before_sampling": True,
    },
    "labels": {
        "scheme": "letras contíguas a partir de A (A, B, C, ...)",
        "translation": "por posição, nunca pelo rótulo textual do dataset original",
        "verified_at_save": True,
        "uniform_num_choices": bool(standard_df["uniforme"].all()),
        "note": (
            "o número de alternativas varia entre itens e datasets; o extrator "
            "aceita só as letras do próprio item e a análise pondera o chute por "
            "num_choices"
        ),
    },
    "per_dataset": {
        r["dataset"]: {
            "problem_type": r["tipo"],
            "native_mcq": bool(r["MCQ nativo"]),
            "num_choices": int(r["alternativas"]),
            "num_choices_distribution": {
                str(k): v for k, v in sorted(
                    __import__("collections").Counter(
                        i["num_choices"]
                        for split in DATASETS[r["dataset"]].splits
                        for i in read_jsonl(SPLITS_DIR / r["dataset"] / f"{split}.jsonl")
                    ).items()
                )
            },
            "train_population_raw": processed_counts[(r["dataset"], "train")],
            "train_population_dedup": len(train_pool[r["dataset"]]),
            "train_selected": int(r["train"]),
            "validation": int(r["validation"]),
            "test": int(r["test"]),
        }
        for r in summary
    },
    "totals": {
        "train_selected": int(summary_df["train"].sum()),
        "validation": int(summary_df["validation"].sum()),
        "test": int(summary_df["test"].sum()),
    },
}

(SPLITS_DIR / "manifest_splits.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)

print(f"treino selecionado : {manifest['totals']['train_selected']:,}")
print(f"validação          : {manifest['totals']['validation']:,}")
print(f"teste              : {manifest['totals']['test']:,}")
print(f"\nmanifesto: {SPLITS_DIR / 'manifest_splits.json'}")
summary_df

treino selecionado : 1,763
validação          : 2,615
teste              : 4,801

manifesto: /home/rodrigo.flexa/Reflection-MCQ/data/splits/manifest_splits.json


,dataset,tipo,MCQ nativo,alternativas,train,validation,test
0,gsm8k,process,False,4,366,0,1319
1,aqua,process,True,5,383,252,246
2,logiqa2,process,True,4,372,1565,1565
3,arc,knowledge,True,4,286,298,1171
4,openbookqa,knowledge,True,4,356,500,500


## O que ficou pronto e o que vem depois

`data/splits/<dataset>/{train,validation,test}.jsonl` é a entrada canônica de
todo o resto do pipeline. Treino selecionado por Cochran; validação e teste
oficiais, inteiros.

Próximo passo: `02_teste_inferencia.ipynb`, que confirma que um modelo consegue
ler esses arquivos, responder no formato exigido e ter a resposta extraída.

Três decisões deste notebook precisam entrar no texto do paper:

- **O GSM8K é um GSM8K-MCQ**, com distratores derivados dos passos intermediários
  anotados no dataset. Números não são comparáveis com GSM8K aberto.
- **Os tamanhos de treino vêm de Cochran com correção de população finita** e
  substituem os valores anteriores do Caderno.
- **O treino foi deduplicado contra validação e teste.** Sem isso, cerca de 40%
  das questões de teste do LogiQA2 teriam um par idêntico no treino, e a
  recuperação por similaridade devolveria a reflexão da questão idêntica — o
  mesmo confundidor de memorização que o paper anterior tinha, agora escondido
  atrás de um cosseno de 1,0. Registrar isso como parte do protocolo, não como
  detalhe de implementação.

Uma limitação a mencionar, sem solução automática: o LogiQA 2.0 tem um número
pequeno de itens em que a premissa não corresponde à pergunta, defeito presente
no release oficial (conferimos que o mirror em parquet é fiel ao original). Não
existe detector confiável para isso — heurísticas de sobreposição lexical
confundem esses itens com as questões de analogia, que legitimamente tratam de
outro assunto que a premissa. Como a amostra de treino do LogiQA2 tem só 372
itens, a inspeção manual é viável e é a recomendação.